## Example 1: Basic property-based testing with @given decorator

In [ ]:
from hypothesis import given, strategies as st

# Example 1: Basic property-based testing with @given decorator
print("Example 1: Basic property-based testing")

@given(x=st.integers(), y=st.integers())
def test_addition_commutative(x, y):
    """Test that addition is commutative"""
    result1 = x + y
    result2 = y + x
    assert result1 == result2
    # A @given test must return None — hypothesis raises `FailedHealthCheck` on a return value,
    # because returning something usually means the author meant to assert it.

# A `@given`-decorated function takes NO arguments from the caller — hypothesis supplies them.
# Calling it as `test_addition_commutative(x, y)` raises
# `TypeError: ... takes 0 positional arguments but 2 were given`. Keep the hand-picked cases on a
# plain helper, and run the property test the way hypothesis intends: with no arguments at all.
def check_addition_commutative(x, y):
    """The property itself, callable directly for illustrative cases."""
    assert x + y == y + x
    return x + y == y + x

print("Testing addition commutativity property...")
try:
    # Illustrative hand-picked cases
    test_cases = [(1, 2), (5, -3), (0, 10), (-7, -2)]
    for x, y in test_cases:
        result = check_addition_commutative(x, y)
        print(f"  {x} + {y} == {y} + {x}: {result}")
    
    # And the real property test: hypothesis generates the inputs itself.
    test_addition_commutative()
    print("  hypothesis generated and checked its own examples")
    
    print("Property testing completed successfully")
except Exception as e:
    print(f"Test completed with note: {e}")
    raise AssertionError(
        f"hypothesis example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Basic property-based testing example completed")

## Example 2: Data generation strategies

In [ ]:
from hypothesis import strategies as st

# Example 2: Data generation strategies
print("Example 2: Data generation strategies")

# Generate different types of data
integer_strategy = st.integers(min_value=1, max_value=100)
text_strategy = st.text(min_size=1, max_size=10)
list_strategy = st.lists(st.integers(), min_size=1, max_size=5)

print("Generated sample data:")

# Generate integers
print("Integers (1-100):")
for _ in range(3):
    sample = integer_strategy.example()
    print(f"  {sample}")

# Generate text
print("Text strings:")
for _ in range(3):
    sample = text_strategy.example()
    print(f"  '{sample}'")

# Generate lists
print("Integer lists:")
for _ in range(3):
    sample = list_strategy.example()
    print(f"  {sample}")

# example() draws at random, so check the bounds each strategy promises
assert all(1 <= integer_strategy.example() <= 100 for _ in range(10))
assert all(1 <= len(text_strategy.example()) <= 10 for _ in range(10))
assert all(1 <= len(list_strategy.example()) <= 5 for _ in range(10))

# find() searches for an example that satisfies a condition and shrinks it to the simplest one,
# so its result is deterministic
from hypothesis import find, settings
no_db = settings(database=None)
assert find(integer_strategy, lambda n: n > 50, settings=no_db) == 51
assert find(list_strategy, lambda xs: sum(xs) > 10, settings=no_db) == [11]
assert len(find(text_strategy, lambda s: True, settings=no_db)) == 1

## Example 3: Using assume() for conditional testing

In [ ]:
from hypothesis import assume, given, strategies as st
import math

# Example 3: Using assume() for conditional testing
print("Example 3: Conditional testing with assume()")

# Bounded integers: a float quotient cannot round-trip an arbitrarily large int, so the unbounded
# strategy would (correctly) find a counterexample to the property below.
@given(x=st.integers(min_value=-10**6, max_value=10**6), y=st.integers(min_value=-10**6, max_value=10**6))
def test_division_with_assume(x, y):
    """Test division with assumption that divisor is not zero"""
    assume(y != 0)  # Skip test cases where y is zero
    # Property: dividing and then multiplying should give back original (within float precision)
    reconstructed = (x / y) * y
    assert math.isclose(reconstructed, x, rel_tol=1e-9, abs_tol=1e-9), (x, y, reconstructed)
    # A @given test must return None, and it takes NO arguments from the caller.

def check_division(x, y):
    """The property itself, callable directly for hand-picked non-zero divisors."""
    result = x / y
    return result, math.isclose(result * y, x, rel_tol=1e-9, abs_tol=1e-9)

# Test the property manually with valid inputs
print("Testing division with non-zero divisors:")
test_cases = [(10, 2), (15, 3), (7, -2), (100, 4)]

for x, y in test_cases:
    result, is_close = check_division(x, y)
    assert is_close, (x, y, result)
    print(f"  {x} / {y} = {result:.2f}, reconstruction check: {is_close}")

# And the real property test: hypothesis generates (x, y) itself; assume() discards y == 0.
test_division_with_assume()
print("  hypothesis generated and checked its own examples")

print("Conditional testing with assume() completed successfully")

## Example 4: Composite strategies and custom data generation

In [ ]:
from hypothesis import strategies as st
from hypothesis.strategies import composite

# Example 4: Composite strategies and custom data generation
print("Example 4: Composite strategies")

@composite
def coordinates(draw):
    """Generate coordinate pairs (x, y) within bounds"""
    x = draw(st.floats(min_value=-10.0, max_value=10.0))
    y = draw(st.floats(min_value=-10.0, max_value=10.0))
    return (x, y)

@composite
def person_data(draw):
    """Generate person data with name and age"""
    name = draw(st.text(min_size=1, max_size=10, alphabet=st.characters(whitelist_categories=('Lu', 'Ll'))))
    age = draw(st.integers(min_value=0, max_value=120))
    return {"name": name, "age": age}

# Generate sample data
print("Generated coordinate pairs:")
for _ in range(3):
    coord = coordinates().example()
    print(f"  ({coord[0]:.2f}, {coord[1]:.2f})")

print("\nGenerated person data:")
for _ in range(3):
    person = person_data().example()
    print(f"  Name: {person['name']}, Age: {person['age']}")

for _ in range(10):
    x, y = coordinates().example()
    assert -10.0 <= x <= 10.0 and -10.0 <= y <= 10.0
    person = person_data().example()
    assert 0 <= person["age"] <= 120 and 1 <= len(person["name"]) <= 10 and person["name"].isalpha()

# Shrinking also works through composite strategies: the simplest adult is 18 with a one-letter name
from hypothesis import find, settings
adult = find(person_data(), lambda p: p["age"] >= 18, settings=settings(database=None))
assert adult["age"] == 18 and len(adult["name"]) == 1

## Example 5: Test configuration and settings

In [ ]:
from hypothesis import settings, Verbosity, strategies as st

# Example 5: Test configuration and settings
print("Example 5: Test configuration and settings")

# Define a function to test list properties
def check_list_properties(numbers):
    """Test basic properties of integer lists"""
    # Property: length of list should be positive
    length_positive = len(numbers) > 0
    
    # Property: max should be >= min
    list_max = max(numbers)
    list_min = min(numbers)
    max_ge_min = list_max >= list_min
    
    # Property: all numbers should be in valid range
    all_in_range = all(1 <= n <= 100 for n in numbers)
    
    return length_positive, max_ge_min, all_in_range

# Show hypothesis settings configuration
print("Hypothesis settings demonstration:")
print(f"Default verbosity: {Verbosity.normal}")
print(f"Available verbosity levels: {list(Verbosity)}")

# Test manually with some examples
print("Testing list properties:")
test_lists = [[1, 2, 3], [50], [10, 20, 30, 5], [100, 1, 50]]

for test_list in test_lists:
    length_pos, max_ge_min, in_range = check_list_properties(test_list)
    print(f"  {test_list}: length>0={length_pos}, max>=min={max_ge_min}, in_range={in_range}")

for test_list in test_lists:
    assert check_list_properties(test_list) == (True, True, True)

assert settings.default.max_examples == 100

# @settings controls how many examples a @given test runs
from hypothesis import given
seen = []

@settings(max_examples=25, database=None, verbosity=Verbosity.quiet)
@given(st.lists(st.integers(min_value=1, max_value=100), min_size=1))
def test_list_properties(numbers):
    seen.append(numbers)
    assert check_list_properties(numbers) == (True, True, True)

test_list_properties()
print(f"Examples run under max_examples=25: {len(seen)}")
assert 0 < len(seen) <= 25